# Contextual Compression Retriever — Advanced RAG Series (2026 Edition)
### by Mayank Chugh · @itaienthusiast · Updated October 2026

**What this notebook shows:** a normal FAISS retriever returns whole chunks; a `ContextualCompressionRetriever` trims or drops those chunks *before* they reach the LLM. We test four compressors — `LLMChainExtractor`, `LLMChainFilter`, `EmbeddingsFilter`, and a `DocumentCompressorPipeline` with `EmbeddingsRedundantFilter` — and measure what each one actually removes.

**Corpus:** the 2022 State of the Union speech (`state_of_the_union.txt`, 95 chunks at 500/100).

#### October 2026 update — what changed vs the original notebook

| Component | This notebook | Why |
|---|---|---|
| Chains / retrievers imports | `langchain_classic.chains`, `langchain_classic.retrievers` | Removed from `langchain` 1.x — the original cells raise `ModuleNotFoundError` |
| Embeddings | `BAAI/bge-small-en-v1.5` (local) | No API key needed; but thresholds must be re-tuned |
| LLM | Local `Qwen/Qwen2.5-3B-Instruct` → Anthropic → OpenAI | Local-first fallback order |
| `EmbeddingsFilter` | Explicit threshold chosen from printed scores | Default is `k=20, similarity_threshold=None` → keeps every doc |
| Ratio cell | Guarded: prints "0 docs kept" | Empty result printed a ratio of ~181,700,000x |
| EmbeddingsFilter length cell | Measures its own result | Original measured the extractor's output by mistake |

> 📊 **Measured in a test run (CPU, bge-small, Qwen2.5-3B-Instruct, greedy decoding):**
> Top-3-priorities query → original 1,817 chars · EmbeddingsFilter default 1,817 (1.00x) · EmbeddingsFilter @0.65 415 (4.38x) · Pipeline @0.60 772 (2.35x)
> Ketanji Brown Jackson query → LLMChainExtractor 1,668 → 330 chars (5.05x)
> Climate query → LLMChainExtractor kept **0 of 4** docs with the local 3B model — see the sanity-check cell.
> Re-confirm these numbers on your T4 run; fp16 on GPU can move similarity scores in the third decimal.

**Runtime:** Colab **T4 GPU** (for the local LLM). Without a GPU the notebook falls back to Anthropic, then OpenAI, if those keys are in Colab Secrets.


## 1. Install

In [ ]:
!pip install -q langchain langchain-classic langchain-community langchain-text-splitters \
             langchain-huggingface langchain-anthropic langchain-openai \
             faiss-cpu sentence-transformers

## 2. Imports

In [ ]:
import os, torch
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.chains import RetrievalQA
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import (
    LLMChainExtractor, LLMChainFilter, EmbeddingsFilter, DocumentCompressorPipeline)
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_community.utils.math import cosine_similarity

## 3. Load the speech

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/langchain-ai/langchain/v0.1.0/docs/docs/modules/state_of_the_union.txt"
if not os.path.exists("state_of_the_union.txt"):
    !wget -q -O state_of_the_union.txt {DATA_URL}
documents = TextLoader("state_of_the_union.txt").load()
print(len(documents[0].page_content), "characters")

## 4. Chunk

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)
print(len(texts), "chunks")

## 5. Embeddings + FAISS retriever

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True})
retriever = FAISS.from_documents(texts, embeddings).as_retriever(search_kwargs={"k": 4})

## 6. Helpers

In [6]:
def pretty_print_docs(docs):
    if not docs:
        print("⚠️  0 documents returned")
        return
    print(f"\n{'-' * 100}\n".join(
        f"Document {i+1}:\n\n{d.page_content}" for i, d in enumerate(docs)))

def context_len(docs):
    return len("\n\n".join(d.page_content for d in docs))

def report(label, original_docs, compressed_docs):
    o, c = context_len(original_docs), context_len(compressed_docs)
    if c == 0:
        print(f"{label}: {o} -> 0 chars (0 docs kept — nothing reaches the LLM)")
    else:
        print(f"{label}: {o} -> {c} chars | {len(compressed_docs)} docs | ratio {o/c:.2f}x")

def show_scores(query, docs):
    q = embeddings.embed_query(query)
    d = embeddings.embed_documents([x.page_content for x in docs])
    for i, s in enumerate(cosine_similarity([q], d)[0]):
        print(f"Doc {i+1}: {s:.4f}  {docs[i].page_content[:70]!r}")

## 7. Basic retrieval — three queries

In [ ]:
Q_KBJ = "What did the president say about Ketanji Brown Jackson"
Q_TOP3 = "What were the top three priorities outlined in the most recent State of the Union address?"
Q_CLIMATE = "How did the President propose to tackle the issue of climate change?"

docs = retriever.invoke(Q_KBJ)
pretty_print_docs(docs)

In [ ]:
docs2 = retriever.invoke(Q_TOP3)
pretty_print_docs(docs2)

In [ ]:
docs3 = retriever.invoke(Q_CLIMATE)
pretty_print_docs(docs3)

In [ ]:
for q, d in [(Q_KBJ, docs), (Q_TOP3, docs2), (Q_CLIMATE, docs3)]:
    print(f"{context_len(d)} chars  <- {q}")
    show_scores(q, d)
    print()

## 8. Choose the LLM (local first → Anthropic → OpenAI)

In [ ]:
LOCAL_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ANTHROPIC_MODEL = "claude-haiku-5-5"
OPENAI_MODEL = "gpt-4o-mini"

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

def build_local():
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
    from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
    tok = AutoTokenizer.from_pretrained(LOCAL_MODEL)
    model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype=torch.float16, device_map="auto")
    pipe = pipeline("text-generation", model=model, tokenizer=tok,
                    max_new_tokens=256, do_sample=False, return_full_text=False)
    return ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe), tokenizer=tok)

def build_api():
    if _secret("ANTHROPIC_API_KEY"):
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=_secret("ANTHROPIC_API_KEY"), temperature=0), f"anthropic:{ANTHROPIC_MODEL}"
    if _secret("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=OPENAI_MODEL, api_key=_secret("OPENAI_API_KEY"), temperature=0), f"openai:{OPENAI_MODEL}"
    return None, None

if torch.cuda.is_available():
    llm, LLM_NAME = build_local(), f"local:{LOCAL_MODEL}"
else:
    llm, LLM_NAME = build_api()
    assert llm is not None, "No GPU and no ANTHROPIC_API_KEY / OPENAI_API_KEY in Colab Secrets"
print("Answer LLM:", LLM_NAME)

## 9. Baseline RAG — no compression

In [ ]:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
print(chain.invoke(Q_TOP3)["result"])

## 10. LLMChainExtractor — keep only the relevant sentences

In [13]:
COMPRESSOR_PREFERENCE = "local"

if COMPRESSOR_PREFERENCE == "api":
    compressor_llm, COMPRESSOR_NAME = build_api()
else:
    compressor_llm, COMPRESSOR_NAME = llm, LLM_NAME
print("Compressor LLM:", COMPRESSOR_NAME)

compressor = LLMChainExtractor.from_llm(compressor_llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever)

Compressor LLM: local:Qwen/Qwen2.5-3B-Instruct


In [ ]:
compressed_docs = compression_retriever.invoke(Q_KBJ)
pretty_print_docs(compressed_docs)
report("Extractor / KBJ", docs, compressed_docs)

In [ ]:
compressed_top3 = compression_retriever.invoke(Q_TOP3)
pretty_print_docs(compressed_top3)
report("Extractor / top-3", docs2, compressed_top3)

In [ ]:
compressed_climate = compression_retriever.invoke(Q_CLIMATE)
pretty_print_docs(compressed_climate)
report("Extractor / climate", docs3, compressed_climate)

### Sanity check — did the extractor throw away relevant text?

In [ ]:
for name, q, before, after in [("KBJ", Q_KBJ, docs, compressed_docs),
                               ("top-3", Q_TOP3, docs2, compressed_top3),
                               ("climate", Q_CLIMATE, docs3, compressed_climate)]:
    flag = "  ⚠️ EMPTY — the answer LLM will get no context" if not after else ""
    print(f"{name:8s} {len(before)} docs -> {len(after)} docs{flag}")

## 11. LLMChainFilter — keep or drop whole documents

In [ ]:
llm_filter = LLMChainFilter.from_llm(compressor_llm)
compression_retriever2 = ContextualCompressionRetriever(
    base_compressor=llm_filter, base_retriever=retriever)
compressed_docs3 = compression_retriever2.invoke(Q_TOP3)
pretty_print_docs(compressed_docs3)
report("LLMChainFilter / top-3", docs2, compressed_docs3)

## 12. EmbeddingsFilter — no LLM, just cosine similarity

In [ ]:
embeddings_filter_default = EmbeddingsFilter(embeddings=embeddings)
print("defaults -> k =", embeddings_filter_default.k,
      "| similarity_threshold =", embeddings_filter_default.similarity_threshold)
compression_retriever3 = ContextualCompressionRetriever(
    base_compressor=embeddings_filter_default, base_retriever=retriever)
compressed_docs4 = compression_retriever3.invoke(Q_TOP3)
report("EmbeddingsFilter default / top-3", docs2, compressed_docs4)

In [20]:
for t in [0.50, 0.60, 0.65, 0.70, 0.76]:
    f = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=t)
    out = ContextualCompressionRetriever(base_compressor=f, base_retriever=retriever).invoke(Q_TOP3)
    report(f"threshold {t:.2f}", docs2, out)

threshold 0.50: 1817 -> 1817 chars | 4 docs | ratio 1.00x
threshold 0.60: 1817 -> 1817 chars | 4 docs | ratio 1.00x
threshold 0.65: 1817 -> 415 chars | 1 docs | ratio 4.38x
threshold 0.70: 1817 -> 0 chars (0 docs kept — nothing reaches the LLM)
threshold 0.76: 1817 -> 0 chars (0 docs kept — nothing reaches the LLM)


In [ ]:
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.65)
compression_retriever3 = ContextualCompressionRetriever(
    base_compressor=embeddings_filter, base_retriever=retriever)
compressed_docs4 = compression_retriever3.invoke(Q_TOP3)
pretty_print_docs(compressed_docs4)
report("EmbeddingsFilter @0.65 / top-3", docs2, compressed_docs4)

## 13. DocumentCompressorPipeline — split, de-duplicate, filter

In [ ]:
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0, separator=". ")
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)
relevant_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.60)
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[splitter, redundant_filter, relevant_filter])
pipeline_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor, base_retriever=retriever)
compressed_pipe = pipeline_retriever.invoke(Q_TOP3)
pretty_print_docs(compressed_pipe)
report("Pipeline @0.60 / top-3", docs2, compressed_pipe)

## 14. Final RAG with the compressed retriever

In [ ]:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=pipeline_retriever)
print(chain.invoke(Q_TOP3)["result"])

## 15. Summary

In [ ]:
srows = [("Original (top-k=4)", docs2), ("LLMChainExtractor", compressed_top3),
        ("LLMChainFilter", compressed_docs3), ("EmbeddingsFilter @0.65", compressed_docs4),
        ("Pipeline @0.60", compressed_pipe)]
base = context_len(docs2)
print(f"{'Method':26s}{'docs':>6s}{'chars':>8s}{'ratio':>9s}")
for name, d in rows:
    c = context_len(d)
    print(f"{name:26s}{len(d):6d}{c:8d}{(f'{base/c:.2f}x' if c else '—'):>9s}")